In [1]:
pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torch.nn as nn
from torchvision import models
import pennylane as qml
import numpy as np
from torchinfo import summary

# ==================== CONFIG ====================
N_QUBITS = 8
N_CLASSES = 9  # Based on your previous output
# Use 'r' before the string to prevent escape character errors in Windows paths
MODEL_PATH = r"E:\Project_with_hybrid\appleleaf.v2i.yolo26-project\models\QResnet\hybrid_quantum_resnet_best.pth"

# ==================== QUANTUM LAYER ====================
dev_q = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_q, interface="torch")
def qnode(inputs, weights):
    """Enhanced quantum circuit"""
    # Angle embedding
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    # Strongly entangling layers
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    # Measurements
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (4, N_QUBITS, 3)}  # Deeper: 4 layers
q_layer = qml.qnn.TorchLayer(qnode, weight_shapes)

class QuantumFiLM(nn.Module):
    """Quantum Feature-wise Linear Modulation"""
    def __init__(self, n_qubits=N_QUBITS):
        super().__init__()
        self.n_qubits = n_qubits
        
        # ResNet50 layer2 output has 512 channels
        self.in_channels = 512 
        
        # Attention mechanism
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(self.in_channels, n_qubits),
            nn.Tanh()
        )
        
        # Quantum-to-classical mapping
        self.fc = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, self.in_channels * 2) 
        )

    def forward(self, x):
        B, C, H, W = x.shape 
        
        # Extract quantum features
        pooled = self.attention(x)
        pooled = pooled * np.pi  # Scale to quantum range
        
        # Quantum processing
        q_out = q_layer(pooled).float()
        
        # Generate scale and shift
        film_params = self.fc(q_out)
        scale, shift = torch.chunk(film_params, 2, dim=1)
        
        # Apply FiLM
        scale = scale.view(B, C, 1, 1)
        shift = shift.view(B, C, 1, 1)
        
        return x * (1 + scale.tanh()) + shift.tanh()

# ==================== HYBRID MODEL ====================
class HybridQuantumResNet(nn.Module):
    """Hybrid Quantum-Classical ResNet"""
    def __init__(self, n_classes=9, n_qubits=N_QUBITS):
        super().__init__()
        
        # Load pretrained ResNet (set to False here just to build the skeleton quickly)
        self.backbone = models.resnet50(pretrained=False)
        
        # Insert quantum layer after layer2
        self.quantum_film = QuantumFiLM(n_qubits)
        
        # Extract ResNet layers
        self.layer1 = nn.Sequential(
            self.backbone.conv1,
            self.backbone.bn1,
            self.backbone.relu,
            self.backbone.maxpool,
            self.backbone.layer1,
            self.backbone.layer2
        )
        
        self.layer2 = nn.Sequential(
            self.backbone.layer3,
            self.backbone.layer4
        )
        
        # Enhanced classification head
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.quantum_film(x)
        x = self.layer2(x)
        x = self.avgpool(x)
        x = self.head(x)
        return x

# ==================== SUMMARY GENERATOR ====================
def generate_summary(pth_path):
    print(f"Initializing QResNet architecture...")
    model = HybridQuantumResNet(n_classes=N_CLASSES, n_qubits=N_QUBITS)
    
    print(f"Attempting to load weights from: {pth_path}")
    try:
        checkpoint = torch.load(pth_path, map_location=torch.device('cpu'))
        if 'model_state' in checkpoint:
            model.load_state_dict(checkpoint['model_state'])
            print(f"Weights loaded successfully! (Saved Val Acc: {checkpoint.get('val_acc', 'Unknown'):.2f}%)")
        else:
            model.load_state_dict(checkpoint)
            print("Weights loaded successfully!")
    except Exception as e:
        print(f"\nWarning: Could not load the .pth file. Error: {e}")
        print("Generating architecture summary using random initial weights instead...\n")
        
    print("\n" + "="*70)
    print("                           QRESNET PARAMETER SUMMARY")
    print("="*70)
    
    # 1, 3, 224, 224 represents Batch Size of 1, 3 RGB Channels, 224x224 Resolution
    summary(
        model, 
        input_size=(1, 3, 224, 224), 
        col_names=["kernel_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
    )

if __name__ == "__main__":
    generate_summary(MODEL_PATH)

Initializing QResNet architecture...
Attempting to load weights from: E:\Project_with_hybrid\appleleaf.v2i.yolo26-project\models\QResnet\hybrid_quantum_resnet_best.pth
Weights loaded successfully! (Saved Val Acc: 98.88%)

                           QRESNET PARAMETER SUMMARY
